# Phishing Detection MVP - Sprint 6

Prototipo Voila del MVP para inferencia local, Business Value, trazabilidad MLflow y estado MongoDB.

In [ ]:
from pathlib import Path
from io import BytesIO

import joblib
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

ROOT = Path.cwd()
MODEL_PATH = ROOT / 'models' / 'final_model.pkl'
TEST_PATH = ROOT / 'data' / 'processed' / 'test.csv'
BUSINESS_SUMMARY_PATH = ROOT / 'reports' / 'business_value_summary.csv'
MLFLOW_SUMMARY_PATH = ROOT / 'reports' / 'mlflow_tracking_summary.csv'
MONGODB_SUMMARY_PATH = ROOT / 'reports' / 'mongodb_export_summary.csv'

TARGET_COL = 'Result'
PHISHING_MODEL_LABELS = (-1, 0)

display(HTML('''
<style>
.mvp-title {font-size: 30px; font-weight: 700; margin-bottom: 4px;}
.mvp-subtitle {font-size: 15px; color: #555; margin-bottom: 18px;}
.status-ok {color: #1b7f37; font-weight: 700;}
.status-warn {color: #9a6700; font-weight: 700;}
.status-error {color: #cf222e; font-weight: 700;}
</style>
<div class='mvp-title'>Phishing Detection MVP - Sprint 6</div>
<div class='mvp-subtitle'>Panel Voila para scoring local, Business Value, MLflow y MongoDB.</div>
'''))

In [ ]:
def load_business_summary():
    if not BUSINESS_SUMMARY_PATH.exists():
        return {}, 'No se encontro reports/business_value_summary.csv.'
    try:
        df = pd.read_csv(BUSINESS_SUMMARY_PATH)
        if {'item', 'value'}.issubset(df.columns):
            return dict(zip(df['item'], df['value'])), None
        return {}, 'business_value_summary.csv no tiene columnas item/value.'
    except Exception as exc:
        return {}, f'Error leyendo Business Value: {exc}'


def as_float(value, default):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def load_table(path):
    if not path.exists():
        return None, f'No se encontro {path.relative_to(ROOT)}.'
    try:
        df = pd.read_csv(path)
        if df.empty:
            return df, f'{path.relative_to(ROOT)} existe, pero esta vacio.'
        return df, None
    except Exception as exc:
        return None, f'Error leyendo {path.relative_to(ROOT)}: {exc}'


def load_model():
    if not MODEL_PATH.exists():
        return None, f'No se encontro {MODEL_PATH.relative_to(ROOT)}.'
    try:
        return joblib.load(MODEL_PATH), None
    except Exception as exc:
        return None, f'No se pudo cargar el modelo: {exc}'


business_summary, business_warning = load_business_summary()
recommended_threshold = as_float(
    business_summary.get('umbral_recomendado', business_summary.get('umbral_optimo')),
    0.09,
)
model, model_error = load_model()
mlflow_df, mlflow_warning = load_table(MLFLOW_SUMMARY_PATH)
mongodb_df, mongodb_warning = load_table(MONGODB_SUMMARY_PATH)

In [ ]:
def status_html(label, ok, detail):
    css = 'status-ok' if ok else 'status-error'
    return f"<b>{label}:</b> <span class='{css}'>{'OK' if ok else 'ERROR'}</span> - {detail}"


model_status = status_html(
    'Estado del modelo',
    model is not None,
    str(MODEL_PATH.relative_to(ROOT)) if model is not None else model_error,
)
threshold_status = f"<b>Threshold recomendado:</b> {recommended_threshold:.2f}"
business_status = status_html(
    'Business Value',
    not business_warning,
    str(BUSINESS_SUMMARY_PATH.relative_to(ROOT)) if not business_warning else business_warning,
)
display(HTML('<br>'.join([model_status, threshold_status, business_status])))

In [ ]:
use_test_checkbox = widgets.Checkbox(value=True, description='Usar data/processed/test.csv')
upload = widgets.FileUpload(accept='.csv', multiple=False, description='Cargar CSV')
threshold_widget = widgets.FloatSlider(
    value=recommended_threshold,
    min=0.01,
    max=0.99,
    step=0.01,
    description='Threshold',
    readout_format='.2f',
)
run_button = widgets.Button(description='Ejecutar predicciones', button_style='primary')
output = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<h3>Datos de entrada</h3>'),
    use_test_checkbox,
    upload,
    threshold_widget,
    run_button,
    output,
]))

In [ ]:
def get_uploaded_dataframe():
    if not upload.value:
        return None
    value = upload.value
    if isinstance(value, dict):
        file_info = next(iter(value.values()))
    else:
        file_info = value[0]
    content = file_info['content']
    return pd.read_csv(BytesIO(content))


def get_input_dataframe():
    if use_test_checkbox.value:
        if not TEST_PATH.exists():
            raise FileNotFoundError('No se encontro data/processed/test.csv')
        return pd.read_csv(TEST_PATH)
    uploaded = get_uploaded_dataframe()
    if uploaded is None:
        raise ValueError('Seleccione un CSV o active el dataset de prueba.')
    return uploaded


def phishing_probability(model, X):
    if not hasattr(model, 'predict_proba'):
        return None
    proba = model.predict_proba(X)
    classes = list(model.classes_)
    phishing_label = next((label for label in PHISHING_MODEL_LABELS if label in classes), None)
    if phishing_label is None:
        return None
    return proba[:, classes.index(phishing_label)]


def run_predictions(df, threshold):
    if model is None:
        raise RuntimeError(model_error)
    features = df.drop(columns=[TARGET_COL], errors='ignore')
    probabilities = phishing_probability(model, features)
    result = df.copy()
    if probabilities is not None:
        result['phishing_probability'] = probabilities
        result['prediction'] = np.where(probabilities >= threshold, 1, 0)
        result['prediction_label'] = np.where(probabilities >= threshold, 'phishing', 'legitimo')
    else:
        raw = model.predict(features)
        result['prediction'] = np.where(np.isin(raw, PHISHING_MODEL_LABELS), 1, 0)
        result['prediction_label'] = np.where(result['prediction'] == 1, 'phishing', 'legitimo')
        result['phishing_probability'] = np.nan
    return result


def business_kpis(result):
    total = len(result)
    alerts = int((result['prediction_label'] == 'phishing').sum())
    phishing_rate = alerts / total if total else 0
    avg_probability = pd.to_numeric(result['phishing_probability'], errors='coerce').mean()
    rows = [
        ('Total URLs evaluadas', total),
        ('Alertas phishing', alerts),
        ('Tasa de alertas', f'{phishing_rate:.2%}'),
        ('Probabilidad promedio phishing', f'{avg_probability:.2%}' if pd.notna(avg_probability) else 'N/D'),
        ('Ahorro neto recomendado USD', business_summary.get('ahorro_neto_recomendado_usd', 'N/D')),
        ('ROI recomendado', business_summary.get('roi_recomendado', 'N/D')),
        ('Valor por 1000 URLs USD', business_summary.get('valor_por_1000_urls_recomendado_usd', 'N/D')),
    ]
    return pd.DataFrame(rows, columns=['KPI', 'Valor'])

In [ ]:
def on_run_clicked(_):
    with output:
        clear_output()
        try:
            df = get_input_dataframe()
            result = run_predictions(df, threshold_widget.value)
            display(HTML('<h3>Predicciones</h3>'))
            display(result[['prediction', 'prediction_label', 'phishing_probability']].head(100))
            display(HTML('<h3>KPIs de Business Value</h3>'))
            display(business_kpis(result))
        except Exception as exc:
            display(HTML(f"<span class='status-error'>Error:</span> {exc}"))


run_button.on_click(on_run_clicked)

In [ ]:
display(HTML('<h3>Resumen MLflow</h3>'))
if mlflow_df is None:
    display(HTML(f"<span class='status-warn'>{mlflow_warning}</span>"))
elif mlflow_df.empty:
    display(HTML(f"<span class='status-warn'>{mlflow_warning}</span>"))
else:
    status_counts = mlflow_df['status'].value_counts() if 'status' in mlflow_df.columns else pd.Series(dtype=int)
    summary = pd.DataFrame([
        ('Runs registrados', len(mlflow_df)),
        ('Success', int(status_counts.get('success', 0))),
        ('Partial', int(status_counts.get('partial', 0))),
        ('Failed', int(status_counts.get('failed', 0))),
    ], columns=['Metrica', 'Valor'])
    display(summary)
    cols = [c for c in ['model_name', 'run_type', 'sprint', 'status', 'metrics_logged'] if c in mlflow_df.columns]
    display(mlflow_df[cols])

In [ ]:
display(HTML('<h3>Estado MongoDB</h3>'))
if mongodb_df is None:
    display(HTML(f"<span class='status-warn'>{mongodb_warning}</span>"))
elif mongodb_df.empty:
    display(HTML(f"<span class='status-warn'>{mongodb_warning}</span>"))
else:
    if 'status' in mongodb_df.columns:
        display(pd.DataFrame(mongodb_df['status'].value_counts()).reset_index().rename(columns={'index': 'status', 'status': 'count'}))
    if 'notes' in mongodb_df.columns and mongodb_df['notes'].astype(str).str.contains('MONGODB_URI', na=False).any():
        display(HTML("<span class='status-warn'>Exportacion skipped: falta configurar MONGODB_URI.</span>"))
    display(mongodb_df)